# Group Comparison

Flexible group-comparison notebook. Build views by genotype, line, cohort, dataset, or custom combinations, then plot RT, MT, psychometric curves, and JND insets.

## 1. Setup

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

ROOT

## 2. Choose Datasets

`DATASET_SELECTIONS` is the safest option when comparing different lines/cohorts. Set it to `None` to use all combinations of `LINES` and `COHORTS`.

In [ ]:
LINES = ["CNTNAP2"]
COHORTS = ["cohort3"]

# Explicit selections can mix lines/cohorts, e.g.:
# DATASET_SELECTIONS = [("CNTNAP2", "cohort2"), ("CNTNAP2", "cohort3"), ("SHANK3", "cohort1")]
DATASET_SELECTIONS = [("CNTNAP2", "cohort2"), ("CNTNAP2", "cohort3"), ("SHANK3", "cohort1")]

## 3. Choose Comparison

Common presets:

- Same line/cohort, compare genotypes: `COMPARISON = "genotypes"`, `SPLIT_BY = "none"`
- Same line, multiple cohorts, keep cohorts separate: `COMPARISON = "genotypes"`, `SPLIT_BY = "dataset"`
- Same line, multiple cohorts, collapse cohorts by genotype: `COMPARISON = "genotypes"`, `SPLIT_BY = "none"`
- Same genotype across lines/cohorts: `COMPARISON = "datasets"`, `GENOTYPES = ["hom"]`
- Fully custom: `COMPARISON = "custom"` and edit `CUSTOM_SPECS`.

In [ ]:
COMPARISON = "genotypes"  # "genotypes", "datasets", "lines", "cohorts", or "custom"
SPLIT_BY = "dataset"         # for COMPARISON="genotypes": "none", "dataset", "line", or "cohort"
GENOTYPES = ["wt", "het", "hom"]

CUSTOM_SPECS = [
    # {"name": "CNTNAP2 hom collapsed", "line": "CNTNAP2", "genotype": "hom"},
    # {"name": "SHANK3 hom", "line": "SHANK3", "genotype": "hom"},
    # {"name": "CNTNAP2 cohort2 hom", "dataset_key": "CNTNAP2:cohort2", "genotype": "hom"},
    # {"name": "CNTNAP2 cohort3 hom", "dataset_key": "CNTNAP2:cohort3", "genotype": "hom"},
]

## 4. Load Data And Build Views

In [ ]:
from Pipeline.group_comparison import (
    build_group_views,
    build_view_style_maps,
    load_groupcomparison_data,
    summarize_views,
)

data = load_groupcomparison_data(
    lines=LINES,
    cohorts=COHORTS,
    dataset_selections=DATASET_SELECTIONS,
)
df_plot = data["df_plot"]

views = build_group_views(
    df_plot,
    comparison=COMPARISON,
    split_by=SPLIT_BY,
    genotypes=GENOTYPES,
    custom_specs=CUSTOM_SPECS,
)

print("Loaded datasets:", data["selections"])
print("Usable dataset keys:", data["usable_dataset_names"])
view_summary = summarize_views(df_plot, views)
display(view_summary)

view_colors, view_styles = build_view_style_maps(df_plot, views)
print("View colors:", view_colors)
print("View styles:", view_styles)


## 5. Prepare Once

This is the expensive step. Rerun it if you change filters, views, datasets, or overlay settings. You do not need to rerun it just to change `LAYOUT` below.

In [ ]:
import importlib
import GroupComparison.config as gconfig
import GroupComparison.plots as gplots
import Pipeline.group_comparison as gc

LOAD_OVERLAYS = True

importlib.reload(gconfig)
importlib.reload(gplots)
importlib.reload(gc)

cfg = gconfig.GroupComparisonConfig(
    error_mode="individuals",
    skip_psy_fits=(50,),
    psychometric_aggregation="animal_trials",
    summary_abort_types=("Fixation", "MT+", "RT-"),
    ild_shift_for_abl50=True,
)

fcfg = gconfig.FilterConfig(
    training_min=16,
    session_min=0,
    drop_repeat_trials=True,
    session_type_values=[1],
    stim_dur_values=[6000],
    sessiontype_or_stimdur="or",
)
style = gconfig.PlotStyle(title_fs=24, label_fs=25, tick_fs=24, legend_fs=16)

bundle = gc.prepare_flexible_groupcomparison(
    df=df_plot,
    views=views,
    load_overlays=LOAD_OVERLAYS,
    cfg=cfg,
    fcfg=fcfg,
    style=style,
    view_colors=view_colors,
    view_styles=view_styles,
)

print("Filtered rows:", len(bundle["df_filtered"]))
print("Prepared views:", list(bundle["prepared"].keys()))

## 6. Plot From Prepared Data

`LAYOUT = "abls_4x3"` is usually best for direct comparison between views. `LAYOUT = "views_3x3"` gives one row per view.
`LAYOUT = "animals_plus_mean"` makes one panel per animal plus a final average panel for the selected view.

Change `LAYOUT` and rerun this cell without recomputing preparation.

In [ ]:
LAYOUT = "abls_4x3"  # "abls_4x3", "views_3x3", "animals_plus_mean", or "psy_params"
PLOT_VIEW = None  # set to a view name to plot just one group for animals_plus_mean

plot_views = [v for v in views if PLOT_VIEW is None or v.name == PLOT_VIEW]
if not plot_views:
    raise ValueError(f"PLOT_VIEW={PLOT_VIEW!r} did not match any selected view.")

out = gc.plot_flexible_groupcomparison(
    bundle=bundle,
    views=plot_views,
    layout=LAYOUT,
    show=True,
)

out["figures"]

## 7. Summary Metrics and Parameters

In [ ]:
LAYOUT = "summary_metrics"  # or "psy_params" or "summary_aborts" "summary_metrics"
PLOT_VIEW = None

# Reuse the bundle prepared in the cell above.
plot_views = [v for v in views if PLOT_VIEW is None or v.name == PLOT_VIEW]
if not plot_views:
    raise ValueError(f"PLOT_VIEW={PLOT_VIEW!r} did not match any selected view.")

out = gc.plot_flexible_groupcomparison(
    bundle=bundle,
    views=plot_views,
    layout=LAYOUT,
    show=True,
)

out["figures"]

## 8. Example Recipes

### Compare genotypes within one cohort

```python
DATASET_SELECTIONS = [("CNTNAP2", "cohort3")]
COMPARISON = "genotypes"
SPLIT_BY = "none"
GENOTYPES = ["wt", "het", "hom"]
```

### Compare genotypes and keep cohorts separate

```python
DATASET_SELECTIONS = [("CNTNAP2", "cohort2"), ("CNTNAP2", "cohort3")]
COMPARISON = "genotypes"
SPLIT_BY = "dataset"
```

### Collapse cohorts and compare genotypes

```python
DATASET_SELECTIONS = [("CNTNAP2", "cohort2"), ("CNTNAP2", "cohort3")]
COMPARISON = "genotypes"
SPLIT_BY = "none"
```

### Compare the same genotype across datasets

```python
DATASET_SELECTIONS = [("CNTNAP2", "cohort2"), ("CNTNAP2", "cohort3"), ("SHANK3", "cohort1")]
COMPARISON = "datasets"
GENOTYPES = ["hom"]
```

### Custom comparison

```python
COMPARISON = "custom"
CUSTOM_SPECS = [
    {"name": "CNTNAP2 hom all cohorts", "line": "CNTNAP2", "genotype": "hom"},
    {"name": "SHANK3 hom", "line": "SHANK3", "genotype": "hom"},
]
```

## 9. Optional Save For Google Slides

In [ ]:
SAVE_FIGURES = False

def _save_figures(figures, out_dir, prefix):
    for name, fig in figures.items():
        if isinstance(fig, dict):
            subdir = out_dir / name
            subdir.mkdir(parents=True, exist_ok=True)
            _save_figures(fig, subdir, prefix)
        else:
            fig.savefig(out_dir / f"{prefix}_{name}.png", dpi=250, bbox_inches="tight")

if SAVE_FIGURES:
    out_dir = ROOT / "outputs" / "group_comparison"
    out_dir.mkdir(parents=True, exist_ok=True)
    _save_figures(out["figures"], out_dir, f"group_comparison_{LAYOUT}")
    print(out_dir)